In [ ]:
import django

django.setup()

In [ ]:
import logging
import sys

from pokebase import APIMetadata, APIResource
from requests.exceptions import HTTPError

from typing import Any, Optional, TypedDict, Unpack

from django.db import transaction
from django.db.models import Model

from core.services.pokebase import pb
from home.models import PersonalDex, Specimen
from pokedex.models import (
    Move,
    Pokemon,
    PokemonAbility,
    PokemonForm,
    PokemonFormType,
    PokemonMove,
    PokemonMoveVersion,
    PokemonSpecies,
    PokemonSpeciesDexEntry,
    PokemonSpeciesVariety,
    PokemonStat,
    PokemonType,
    Version,
    VersionGameIndex,
    VersionGroup,
)

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
)

logger = logging.getLogger(__name__)


class PokemonBuilder:
    def __get_api_metadata_as_dict(self, api_metadata: APIMetadata) -> dict[str, Any]:
        _ALLOWED_TYPES = (bool, str, int, float, APIMetadata, APIResource)

        if not isinstance(api_metadata, APIMetadata):
            raise Exception(f"objeto informado deve implementar {APIMetadata}")

        result = {}

        for key, value in vars(api_metadata).items():
            if value and not isinstance(value, _ALLOWED_TYPES):
                logger.warning(
                    f"[{key}] value deve ser {_ALLOWED_TYPES} e recebeu {type(value)}"
                )
                continue

            if type(value) == APIMetadata:
                result[key] = self.__get_api_metadata_as_dict(value)
                continue

            result[key] = value

        return result

    def __get_resource_list_names(self, r_list: list) -> list[str]:
        return [obj.name for obj in r_list]

    def __get_object_or_none(self, m: Model, **lkp: dict[str, Any]) -> Optional[Model]:
        try:
            return m.objects.get(**lkp)
        except m.DoesNotExist:
            return None

    def __get_evolves_from_species(self, r: APIResource) -> Optional[PokemonSpecies]:
        if not r.evolves_from_species:
            return None

        return self.__get_object_or_none(
            PokemonSpecies, name=r.evolves_from_species.name
        )

    def __get_pokemon(self, r: APIResource) -> tuple[Pokemon, bool]:
        return Pokemon.objects.get_or_create(
            name=r.name,
            defaults={
                "pokeapi_id": r.id,
                "base_experience": r.base_experience,
                "height": r.height,
                "is_default": r.is_default,
                "order": r.order,
                "weight": r.weight,
                "sprites": self.__get_api_metadata_as_dict(r.sprites),
                "cries": self.__get_api_metadata_as_dict(r.cries),
            },
        )

    def __get_pokemonform(self, r: APIResource) -> tuple[PokemonForm, bool]:
        logger.debug("linking form '%s'" % r.name)
        p_form, created = PokemonForm.objects.get_or_create(
            name=r.name,
            defaults={
                "order": r.order,
                "pokeapi_id": r.id,
                "form_order": r.form_order,
                "is_default": r.is_default,
                "is_battle_only": r.is_battle_only,
                "is_mega": r.is_mega,
                "form_name": r.form_name,
                "sprites": self.__get_api_metadata_as_dict(r.sprites),
                "version_group": self.__get_version_group(str(r.version_group))[0],
            },
        )

        self.__link_pokemon_form_types(r.types, p_form)

        return p_form, created

    def __get_pokemonspecies(self, r: APIResource) -> tuple[PokemonSpecies, bool]:
        logger.debug("linking species: %s" % r.name)
        species, created = PokemonSpecies.objects.get_or_create(
            name=r.name,
            defaults={
                "order": r.order,
                "gender_rate": r.gender_rate,
                "capture_rate": r.capture_rate,
                "base_happiness": r.base_happiness,
                "is_baby": r.is_baby,
                "is_legendary": r.is_legendary,
                "is_mythical": r.is_mythical,
                "hatch_counter": r.hatch_counter,
                "has_gender_differences": r.has_gender_differences,
                "forms_switchable": r.forms_switchable,
                "growth_rate": r.growth_rate,
                "egg_groups": [g.name for g in r.egg_groups],
                "color": r.color,
                "shape": r.shape,
                "evolves_from_species": self.__get_evolves_from_species(r),
                "generation": r.generation,
            },
        )

        return species, created

    def __get_version(self, r: APIResource) -> tuple[Version, bool]:
        return Version.objects.get_or_create(
            name=r.name,
            defaults={
                "version_group": self.__get_version_group(r.version_group.name)[0],
            },
        )

    def __get_version_group(self, name: str) -> tuple[VersionGroup, bool]:
        version_group = pb.APIResource("version-group", name)

        if not version_group:
            return None, False

        return VersionGroup.objects.get_or_create(
            name=name,
            defaults={
                "generation": version_group.generation.name,
                "order": version_group.order,
                "pokedexes": self.__get_resource_list_names(version_group.pokedexes),
                "regions": self.__get_resource_list_names(version_group.regions),
                "versions": self.__get_resource_list_names(version_group.versions),
            },
        )

    def __link_abilities(self, r: APIResource, pokemon: Pokemon):
        for a in r.abilities:
            logger.debug("linking ability: %s" % a.ability)
            pokemon.abilities.add(
                PokemonAbility.objects.get_or_create(
                    slot=a.slot,
                    is_hidden=a.is_hidden,
                    ability=str(a.ability),
                )[0]
            )

    def __link_forms(self, r_forms: list[APIResource], pokemon: Pokemon):
        logger.debug("linking forms for '%s'" % pokemon.name)
        pokemon.forms.add(*[self.__get_pokemonform(f)[0] for f in r_forms])

    def __link_game_indices(self, indices: list[APIResource], pokemon: Pokemon):
        for i in indices:
            logger.debug(
                "linking index %s#%d a %s"
                % (i.version.name, i.game_index, pokemon.name)
            )
            pokemon.game_indices.add(
                VersionGameIndex.objects.get_or_create(
                    version=self.__get_version(i.version)[0],
                    game_index=i.game_index,
                )[0]
            )

    def __link_moves(self, moves: list[APIResource], pokemon: Pokemon):
        all_resource_moves = {m.move for m in moves}
        all_move_names = {str(m) for m in all_resource_moves}
        existing_moves = Move.objects.filter(name__in=all_move_names)
        e_move_names = set(existing_moves.values_list("name", flat=True))
        m_move_names = {mmn for mmn in all_move_names if mmn not in e_move_names}
        r_vgdetails = {str(m): m.version_group_details for m in moves}

        if pokemon.moves.count() == len(all_move_names):
            logger.warning("moves already linked, skipping...")
            return

        logger.debug("%d moves retrieved" % existing_moves.count())
        created_moves = []
        missing_moves = []

        for m in m_move_names:
            r_move = pb.move(m)
            logger.debug("preparing for creating move: %s" % r_move.name)
            missing_moves.append(
                Move(
                    name=r_move.name,
                    accuracy=r_move.accuracy,
                    effect_chance=r_move.effect_chance,
                    pp=r_move.pp,
                    priority=r_move.priority,
                    power=r_move.power,
                    damage_class=r_move.damage_class,
                    generation=r_move.generation,
                    target=r_move.target,
                    type=r_move.type,
                )
            )

        if len(missing_moves):
            created_moves = Move.objects.bulk_create(missing_moves)
            logger.debug("%d moves created" % len(created_moves))

        missing_pmoves = [PokemonMove(move=m) for m in existing_moves]

        if len(missing_pmoves):
            created_pmoves = PokemonMove.objects.bulk_create(missing_pmoves)
            pokemon.moves.add(*created_pmoves)
            logger.debug("%d moves linked" % len(created_pmoves))

        for r_move in all_move_names:
            vg_details = []

            try:
                p_move = pokemon.moves.get(move__name=r_move)
            except:
                print("quebrou em", r_move)
                raise

            for v in r_vgdetails.get(r_move, []):
                vg_details.append(
                    PokemonMoveVersion(
                        version_group=self.__get_version_group(str(v.version_group))[0],
                        move_learn_method=v.move_learn_method,
                        level_learned_at=v.level_learned_at,
                        order=v.order,
                    )
                )

            pm_versions = PokemonMoveVersion.objects.bulk_create(vg_details)
            p_move.version_group_details.add(*pm_versions)

    def __link_pokemon_form_types(self, types: list[APIResource], form: PokemonForm):
        logger.debug("linking '%s' form types" % form.name)
        self.__link_types(PokemonFormType, types, form)

    def __link_pokemon_types(self, types: list[APIResource], pokemon: Pokemon):
        logger.debug("linking '%s' types" % pokemon.name)
        self.__link_types(PokemonType, types, pokemon)

    def __link_dexes_ids(self, numbers: list[APIResource], p_species: PokemonSpecies):
        for pn in numbers:
            logger.debug("linking pokedex: %s#%d" % (str(pn.pokedex), pn.entry_number))
            p_species.pokedex_numbers.add(
                PokemonSpeciesDexEntry.objects.get_or_create(
                    entry_number=pn.entry_number,
                    pokedex=str(pn.pokedex),
                )[0]
            )

    def __link_varieties(self, varieties: list[APIResource], p_species: PokemonSpecies):
        for sv in varieties:
            logger.debug("linking variety: %s" % str(sv.pokemon))
            sv_pokemon = self.__get_object_or_none(Pokemon, name=str(sv.pokemon))

            p_species.varieties.add(
                PokemonSpeciesVariety.objects.get_or_create(
                    is_default=sv.is_default,
                    pokemon=sv_pokemon,
                )[0]
            )

    def __link_species(self, r_species: list[APIResource], pokemon: Pokemon):
        if pokemon.species:
            logger.debug("%s species already linked!" % pokemon.name)
            return

        species, _ = self.__get_pokemonspecies(r_species)

        self.__link_dexes_ids(self.resource.species.pokedex_numbers, species)
        self.__link_varieties(self.resource.species.varieties, species)

        pokemon.species = species
        pokemon.save()

    def __link_stats(self, stats: list[APIResource], pokemon: Pokemon):
        for s in stats:
            logger.debug("linking base %s: %d" % (s.stat, s.base_stat))
            pokemon.stats.add(
                PokemonStat.objects.get_or_create(
                    stat=s.stat,
                    effort=s.effort,
                    base_stat=s.base_stat,
                )[0]
            )

    def __link_types(
        self,
        t_model: PokemonType | PokemonFormType,
        r_types: list[APIResource],
        m_obj: Pokemon | PokemonForm,
    ):
        for t in r_types:
            logger.debug("linking type: %s" % t.type)
            m_obj.types.add(
                t_model.objects.get_or_create(
                    slot=t.slot,
                    type=t.type,
                )[0]
            )

    @transaction.atomic()
    def build_pokemon(self, **kwargs) -> tuple[Pokemon, bool]:
        logger.info(
            "[%d/%d] building Pokemon: %s..."
            % (
                kwargs.get("item", 1),
                kwargs.get("size", 1),
                self.resource.name,
            )
        )
        pokemon, created = self.__get_pokemon(self.resource)

        if kwargs.get("link_abilities", False):
            logger.info("linking abilities...")
            self.__link_abilities(self.resource, pokemon)

        if kwargs.get("link_forms", False):
            logger.info("linking forms...")
            self.__link_forms(self.resource.forms, pokemon)

        if kwargs.get("link_game_indices", False):
            logger.info("linking game_indices...")
            self.__link_game_indices(self.resource.game_indices, pokemon)

        if kwargs.get("link_moves", False):
            logger.info("linking moves...")
            self.__link_moves(self.resource.moves, pokemon)

        if kwargs.get("link_species", False):
            logger.info("linking species...")
            self.__link_species(self.resource.species, pokemon)

        if kwargs.get("link_stats", False):
            logger.info("linking stats...")
            self.__link_stats(self.resource.stats, pokemon)

        if kwargs.get("link_types", False):
            logger.info("linking types...")
            self.__link_pokemon_types(self.resource.types, pokemon)

        return pokemon, created

    @transaction.atomic()
    def build_pokemonform(self, **kwargs) -> tuple[PokemonForm, bool]:
        logger.info(
            "[%d/%d] building PokemonForm: %s..."
            % (
                kwargs.get("item", 1),
                kwargs.get("size", 1),
                self.resource.name,
            )
        )
        return self.__get_pokemonform(self.resource)

    @transaction.atomic()
    def build_pokemonspecies(self, **kwargs) -> tuple[PokemonSpecies, bool]:
        logger.info(
            "[%d/%d] building PokemonSpecies: %s..."
            % (
                kwargs.get("item", 1),
                kwargs.get("size", 1),
                self.resource.name,
            )
        )
        return self.__get_pokemonspecies(self.resource)

    @transaction.atomic()
    def build_version(self, **kwargs) -> tuple[Version, bool]:
        logger.info(
            "[%d/%d] building Version: %s..."
            % (
                kwargs.get("item", 1),
                kwargs.get("size", 1),
                self.resource.name,
            )
        )
        return self.__get_version(self.resource)

    def __init__(self, resource: str | APIResource = "bulbasaur"):
        if isinstance(resource, APIResource):
            self.resource = resource
            return

        self.resource = pb.pokemon(resource)

In [ ]:
class NamedResouceListSyncDict(TypedDict):
    limit: Optional[int]
    offset: Optional[int]
    names: list[Any] = []


class PokemonSyncDict(NamedResouceListSyncDict):
    link_abilities: bool
    link_forms: bool
    link_game_indices: bool
    link_moves: bool
    link_species: bool
    link_stats: bool
    link_types: bool


def __link_existing_specimens(p_dex: PersonalDex):
    slots = p_dex.slot_set.filter(form__isnull=False, specimen__isnull=True)
    s_count = slots.count()

    for i, slot in enumerate(slots):
        specimen = Specimen.objects.filter(form_id=slot.form_id).first()

        if not specimen:
            logger.warning(
                "[%d/%d] no specimen found for %r..." % (i + 1, s_count, slot)
            )
            continue

        logger.info("[%d/%d] linking %r to %r..." % (i + 1, s_count, specimen, slot))
        slot.specimen = specimen
        slot.save()


def __sync_pokemon(**kwargs: Unpack[PokemonSyncDict]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = kwargs.get(
        "names", [n for n in pb.APIResourceList("pokemon").names][offset:limit]
    )

    for index, name in enumerate(names):
        try:
            builder = PokemonBuilder(resource=name)
            builder.build_pokemon(item=index + 1, size=len(names), **kwargs)
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                logger.info(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                logger.warning(
                    f"Erro HTTP {status_code} no ID {name}, mas continuando..."
                )


def __sync_pokemonform(**kwargs: Unpack[NamedResouceListSyncDict]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = kwargs.get(
        "names", [n for n in pb.APIResourceList("pokemon-form").names][offset:limit]
    )

    for index, name in enumerate(names):
        try:
            r_form = pb.pokemon_form(name)

            builder = PokemonBuilder(resource=r_form)
            builder.build_pokemonform(item=index + 1, size=len(names))
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                logger.info(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                logger.warning(
                    f"Erro HTTP {status_code} no ID {name}, mas continuando..."
                )


def __sync_pokemonspecies(**kwargs: Unpack[NamedResouceListSyncDict]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = kwargs.get(
        "names", [n for n in pb.APIResourceList("pokemon-species").names][offset:limit]
    )

    for index, name in enumerate(names):
        try:
            r_species = pb.pokemon_species(name)

            builder = PokemonBuilder(resource=r_species)
            builder.build_pokemonspecies(item=index + 1, size=len(names))
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                logger.info(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                logger.warning(
                    f"Erro HTTP {status_code} no ID {name}, mas continuando..."
                )


def __sync_versions(**kwargs: Unpack[NamedResouceListSyncDict]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = kwargs.get(
        "names", [n for n in pb.APIResourceList("version").names][offset:limit]
    )

    for index, name in enumerate(names):
        try:
            r_version = pb.APIResource("version", name)

            builder = PokemonBuilder(resource=r_version)
            builder.build_version(item=index + 1, size=len(names))
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                logger.info(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                logger.warning(
                    f"Erro HTTP {status_code} no ID {name}, mas continuando..."
                )

In [ ]:
# __sync_versions()
# __sync_pokemonform()
# __sync_pokemonspecies()
# __sync_pokemon(
#     link_abilities=True,
#     link_forms=True,
#     link_game_indices=True,
#     # link_moves=True,
#     link_species=True,
#     link_stats=True,
#     link_types=True,
# )

p_dex = PersonalDex.objects.first()

if p_dex:
    __link_existing_specimens(p_dex)